## 1. 研究動機 (Research Motivation)

建立此基準模型（Baseline Model）的核心目的，是為了提供客觀、可靠的對比標竿。

在導入複雜的深度增強式學習（Deep Reinforcement Learning, DRL）之前，必須先建立穩固的基礎：
1. **確保資料純淨度**：妥善處理歷史數據中的生存者偏誤（Survivorship Bias），確保回測結果貼近真實市場。
2. **建立傳統交易邏輯底盤**：實作具備共整合（Cointegration）與 Z-Score 邏輯的傳統配對交易系統，驗證均值回歸特性的有效性，作為後續 RL 代理人學習的基礎狀態與邏輯參考。

## 2. 文獻回顧 (Literature Review)

配對交易（Pairs Trading）為統計套利（Statistical Arbitrage）中經典的市場中性策略，本階段實作主要基於以下經典框架與最新研究進行建構：

- **傳統配對交易框架**：參考 Gatev, Goetzmann & Rouwenhorst (2006) 提出的經典方法論。該文獻奠定了使用距離法（Sum of Squared Differences, SSD）來篩選潛在配對，並採用滾動視窗（Rolling Window）機制的業界標準。
- **現代特徵擴展與機器學習應用**：參考 Yufei Sun (2025) 對於配對交易系統性文獻的最新整理。其研究確立了將總體經濟指標（Macroeconomic Indicators，例如 VIX）納入考量，以及後續應用機器學習進行動態分群（Clustering）的發展方向，這也成為我們後續強化學習模型的特徵工程藍圖。

## 3. 資料說明與前置處理 (Data Description & Preprocessing)

- **資料來源**：本研究使用 S&P 500 歷史日 K 線資料庫 (`sp500_data.db`)，資料期間涵蓋 **2000-01-01 至 2025-12-31** 的數據。
- **資料清洗步驟**：
    1. **前向填充與剔除稀疏股票**：針對價格序列進行前向填充（Forward-fill）最多 5 天，避免假日或停牌影響，並剔除有效交易日過少的稀疏股票。
    2. **生存者偏誤（Survivorship Bias）處理**：確保資料庫中包含歷史已下市成分股之數據，避免回測時因只考慮現有成分股而產生高估績效的生存者偏誤。
    3. **宏觀指標融合**：自動整合 Yahoo Finance 的 VIX 波動率指數至特徵矩陣中，為未來環境狀態空間（State Space）提供總體經濟環境的降維特徵。

 ## 4. 策略說明與實作細節 (Strategy Description & Implementation Details)

> **策略依據**：Gatev, Goetzmann & Rouwenhorst (2006)

## 策略總覽
配對交易為**市場中性**的統計套利策略。

| 階段 | 名稱 | 說明 |
|:---:|------|------|
| **1** | 初始化與資料收集 | 參數、SQLite、清理 |
| **2** | 配對選擇 | 協整檢驗 → SSD 排序 |
| **3** | 策略執行 | Z-Score 訊號 → PnL |
| **4** | 績效評估 | Sharpe、MDD、CAGR |


### 階段一：初始化與資料收集
### 1-1 套件安裝


In [1]:
import subprocess, sys
for pkg in ['statsmodels','plotly','ipywidgets','kaleido','yfinance']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import warnings; warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
from statsmodels.tsa.stattools import coint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML
pd.set_option('display.float_format','{:.4f}'.format)
# print('套件導入完成。')


### 1-2 全域參數設定

> **使用者只需修改此區塊，即可調整所有策略參數。**

| 參數 | 預設値 | 說明 |
|------|--------|------|
| `DB_PATH` | `sp500_data.db` | SQLite 資料庫路徑 |
| `FORMATION_WINDOW` | `252` | 形成期（12M）|
| `TRADING_WINDOW` | `126` | 交易期（6M）|
| `ROLLING_WINDOW` | `20` | 滾動窗口（1M）|
| `COINT_P_VALUE` | `0.01` | 協整檢驗顯著水準 |
| `TOP_N_PAIRS` | `[5,20]` | Top-N 配對數 |
| `Z_ENTRY` | `1.5` | 進場閾値 |
| `Z_EXIT` | `0.75` | 出場閾値 |
| `STOP_LOSS` | `3.0` | 停損改密 |
| `TRANSACTION_COST` | `0.001` | 單邊成本 0.1% |


In [2]:
# ================================================================
#   Global Parameter Block  -- Edit here to tune strategy
# ================================================================
import os as _os
# DB_PATH: absolute path to the 342MB database (project root)
DB_PATH           = r'c:\Clark\YZU\Papper\Code\sp500_data.db'
START_DATE        = '2000-01-01'
END_DATE          = '2025-12-31'
FORMATION_WINDOW  = 252
TRADING_WINDOW    = 126
ROLLING_WINDOW    = 20
COINT_P_VALUE     = 0.01
TOP_N_PAIRS       = [5, 20]
Z_ENTRY           = 1.5
Z_EXIT            = 0.75
STOP_LOSS         = 3.0
TRANSACTION_COST  = 0.001
SECTOR_NEUTRAL    = True
TARGET_SECTOR     = 'Financials'          # 設定為All為全產業，可指定特定產業
MIN_HISTORY_DAYS  = 200
HEDGE_RATIO       = 1.0
print(f'DB     : {DB_PATH}')
print(f'DB exists: {_os.path.exists(DB_PATH)}')
print(f'DB size  : {_os.path.getsize(DB_PATH)//1024//1024} MB')
print(f'Period : {START_DATE} ~ {END_DATE}')
print(f'Windows: {FORMATION_WINDOW}d / {TRADING_WINDOW}d')


DB     : c:\Clark\YZU\Papper\Code\sp500_data.db
DB exists: True
DB size  : 342 MB
Period : 2000-01-01 ~ 2025-12-31
Windows: 252d / 126d


### 1-3 資料讀取與清理

**清理流程**: 前向填充(最多5天) → 剪除稀疏股票 → 對齊日期

**資料庫結構 (`sp500_data.db`)**:

| 資料表 | 關鍵欄位 |
|--------|----------|
| `daily_prices` | `ticker, date, adj_close` |
| `tickers` | `ticker, sector` |


In [3]:
def load_data_from_db(db_path, start_date, end_date):
    """Load price and sector data from SQLite."""
    conn = sqlite3.connect(db_path)
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'Price rows: {len(prices_df):,}')
                break
        except Exception:
            continue
    if prices_df is None or len(prices_df) == 0:
        tbls = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
        conn.close()
        raise RuntimeError(f"No price table. Tables: {tbls['name'].tolist()}")
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'Sector rows: {len(sector_df):,}')
                break
        except Exception:
            continue
    conn.close()
    if sector_df is None or len(sector_df) == 0:
        print('WARNING: No sector table, using Unknown.')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
        
    # [Mod] Mitigate Survivorship Bias: Check if delisted components exist
    max_date = prices_df['date'].max()
    max_dates = prices_df.groupby('ticker')['date'].max()
    delisted_count = (max_dates < max_date - pd.Timedelta(days=30)).sum()
    if delisted_count < 10:
        import logging
        logging.warning("STRICT RESEARCH LIMITATION: Database lacks historically delisted S&P 500 components. Survivorship Bias is present.")
        
    return prices_df, sector_df


def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """Pivot, forward-fill, drop sparse tickers."""
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    pivot.ffill(limit=5, inplace=True)
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    print(f'Matrix: {len(pivot)} days x {len(pivot.columns)} tickers')

    # [Mod] Integrate Macroeconomic Indicator (VIX)
    import yfinance as yf
    print("Fetching VIX data...")
    try:
        vix_data = yf.download("^VIX", start=pivot.index.min(), end=pivot.index.max() + pd.Timedelta(days=1), progress=False)
        if isinstance(vix_data.columns, pd.MultiIndex):
            vix_close = vix_data['Close'].squeeze()
        else:
            vix_close = vix_data['Close']
        vix_df = pd.DataFrame({'VIX': vix_close})
        vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
        vix_aligned = vix_df.reindex(pivot.index).ffill()
        print("VIX feature matrix aligned.")
    except Exception as e:
        print(f"Error fetching VIX: {e}")
        vix_aligned = pd.DataFrame(index=pivot.index, columns=['VIX'])

    return pivot, vix_aligned


In [4]:
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)
price_pivot, vix_features = preprocess_prices(prices_raw)
sector_map  = sector_info.set_index('ticker')['sector'].to_dict()
display(price_pivot.iloc[:3, :5])
print(pd.Series(sector_map).value_counts().head(10))


Price rows: 3,607,121
Sector rows: 843
Matrix: 6539 days x 660 tickers
Fetching VIX data...
VIX feature matrix aligned.


ticker,A,AA,AAL,AAP,AAPL
date,,,,,
2000-01-03,43.0364,68.8711,NaN,NaN,0.8385
2000-01-04,39.7489,69.1902,NaN,NaN,0.7678
2000-01-05,37.2833,73.1788,NaN,NaN,0.7790


Unknown                   340
Industrials                79
Financials                 76
Information Technology     71
Health Care                60
Consumer Discretionary     48
Consumer Staples           36
Utilities                  31
Real Estate                31
Materials                  26
Name: count, dtype: int64


### 階段二：配對選擇（形成期）

**Engle-Granger 協整檢驗**：
$$P_{A,t} = \\alpha + \\beta P_{B,t} + \\varepsilon_t$$
如果殘差 $\\varepsilon_t$ 的 ADF $p < 0.01$，則判定協整。

**SSD 排序**: $\\text{SSD}(A,B)=\\sum(r_A^{norm}-r_B^{norm})^2$

SSD 越小 → 趨勢越相似 → 優先選入交易期。

**選定流程**: 同產業候選 → 協整檢驗 ($p<0.01$) → SSD由小到大排序 → Top-N


In [5]:
from joblib import Parallel, delayed
import pandas as pd
from itertools import combinations
from statsmodels.tsa.stattools import coint

def normalize_prices(prices):
    return prices / prices.iloc[0]

def compute_ssd(na, nb):
    return ((na - nb)**2).sum()

# --- 新增：獨立的單一配對檢驗邏輯 (用於多核心平行運算) ---
def test_single_coint(a, b, sector, price_window, coint_pval, min_len):
    try:
        sa = price_window[a].dropna()
        sb = price_window[b].dropna()
        idx = sa.index.intersection(sb.index)
        
        if len(idx) < min_len: 
            return None
            
        _, pval, _ = coint(sa[idx].values, sb[idx].values)
        
        if pval < coint_pval:
            return {'stock_a': a, 'stock_b': b, 'sector': sector, 'coint_p': pval}
    except Exception:
        return None
    return None

# --- 修改：主篩選函數 ---
def select_pairs_for_window(price_window, sector_map, top_ns,
                            coint_pval=COINT_P_VALUE,
                            sector_neutral=SECTOR_NEUTRAL,
                            target_sector=TARGET_SECTOR):
    """嚴格遵守文獻：先 Cointegration 檢驗 (平行運算加速) -> 後 SSD 排序"""
    norm = normalize_prices(price_window.dropna(axis=1))
    valid = norm.columns.tolist()
    
    # 產業過濾與分群邏輯保持不變
    if target_sector is not None and target_sector != 'All':
        valid = [t for t in valid if sector_map.get(t, 'Unknown') == target_sector]
        candidates = [(a, b, target_sector) for a,b in combinations(valid, 2)]
    elif sector_neutral:
        groups = {}
        for t in valid:
            groups.setdefault(sector_map.get(t, 'Unknown'), []).append(t)
        candidates = [(a,b,s) for s, ms in groups.items() if len(ms)>=2 for a,b in combinations(ms,2)]
    else:
        candidates = [(a,b, 'All') for a,b in combinations(valid, 2)]
        
    if not candidates: 
        return {n: [] for n in top_ns}

    min_len = FORMATION_WINDOW * 0.8

    # [核心優化] 使用 Joblib 啟動多核心平行運算進行協整檢驗
    coint_results = Parallel(n_jobs=-1, batch_size='auto')(
        delayed(test_single_coint)(a, b, sector, price_window, coint_pval, min_len)
        for a, b, sector in candidates
    )
    
    # 過濾掉未通過檢驗或回傳 None 的結果
    passed = [res for res in coint_results if res is not None]

    if not passed:
        return {n: [] for n in top_ns}

    # 對「通過協整檢驗」的配對計算 SSD
    for p in passed:
        # p['stock_a'] 與 p['stock_b'] 是字串，直接傳入 DataFrame 取欄位
        p['ssd'] = compute_ssd(norm[p['stock_a']], norm[p['stock_b']])
        
    # 依照 SSD 排序並選出 Top-N
    df = pd.DataFrame(passed).sort_values('ssd')
    return {n: df.head(n).to_dict('records') for n in top_ns}

### 階段三：策略執行（交易期）

$$Z_t = (\\text{spread}_t - \\mu_{hist})/\\sigma_{hist}$$

| 訊號 | 操作 |
|------|---------|
| $Z > +1.5$ | 放空A、做多B |
| $Z < -1.5$ | 做多A、放空B |
| $Z(絕對值) < 0.75$ | 平倉 |
| $Z(絕對值) > 3.0$ | 停損平倉 |

**跨日執行法 (Wait-One-Day)**: 訊號後隔日執行，避免 bid-ask bounce。
**Dollar-Neutral**: 每配對 $1 做多 + $1 放空。


In [6]:
def execute_pair_trades(trade_prices, pair, form_prices,
                        z_entry=Z_ENTRY, z_exit=Z_EXIT,
                        stop_loss=STOP_LOSS, cost=TRANSACTION_COST,
                        hedge_ratio=HEDGE_RATIO):
    """Dollar-neutral pair trade with wait-one-day rule (NumPy Optimized)."""
    a, b = pair['stock_a'], pair['stock_b']
    try:
        pa_t = trade_prices[a].dropna()
        pb_t = trade_prices[b].dropna()
        pa_f = form_prices[a].dropna()
        pb_f = form_prices[b].dropna()
    except KeyError:
        return pd.Series(0.0, index=trade_prices.index)

    common = pa_t.index.intersection(pb_t.index)
    if len(common) < 5:
        return pd.Series(0.0, index=trade_prices.index)

    na = pa_t[common] / pa_f.iloc[-1]
    nb = pb_t[common] / pb_f.iloc[-1]

    fi = pa_f.index.intersection(pb_f.index)
    fs = (pa_f[fi]/pa_f[fi].iloc[0]) - (pb_f[fi]/pb_f[fi].iloc[0])
    mu, sd = fs.mean(), fs.std()

    if sd < 1e-8:
        return pd.Series(0.0, index=trade_prices.index)

    z = ((na - nb) - mu) / sd

    # 提前轉換為 NumPy Array 以避開 Pandas 迴圈延遲
    z_vals = z.values
    na_vals = na.values
    nb_vals = nb.values

    pos, p = 0.0, 0.0
    pna, pnb = na_vals[0], nb_vals[0]
    pnl_d = {}

    for i in range(len(common)):
        zi, cna, cnb = z_vals[i], na_vals[i], nb_vals[i]
        dt = common[i]

        if i > 0 and pos != 0:
            p = pos * ((cna - pna)/pna - hedge_ratio * (cnb - pnb)/pnb)
        else:
            p = 0.0

        new = pos
        if pos == 0:
            if zi > z_entry:
                new = -1
                p -= 2 * cost * (1 + hedge_ratio)
            elif zi < -z_entry:
                new = +1
                p -= 2 * cost * (1 + hedge_ratio)
        else:
            if abs(zi) < z_exit or abs(zi) > stop_loss:
                p -= 2 * cost * (1 + hedge_ratio)
                new = 0

        pos, pna, pnb = new, cna, cnb
        pnl_d[dt] = p

    return pd.Series(pnl_d, dtype=float).reindex(trade_prices.index, fill_value=0.0)

### 滾動視窗主迴圈

```
視窗 0: 形成期 [Day 0..251]  | 交易期 [Day 252..377]
視窗 1: 形成期 [Day 126..377] | 交易期 [Day 378..503]
(每次向前滾動 ROLLING_WINDOW 個交易日)
```


In [7]:
def run_backtesting(price_pivot, sector_map,
                    formation_window=FORMATION_WINDOW,
                    trading_window=TRADING_WINDOW,
                    rolling_window=ROLLING_WINDOW,
                    top_ns=TOP_N_PAIRS):
    """Main rolling-window backtest loop."""
    dates = price_pivot.index
    N     = len(dates)
    all_pnl = {n: pd.DataFrame(index=dates) for n in top_ns}
    records = []
    si, wid = 0, 0
    while si + formation_window + trading_window <= N:
        fe = si + formation_window
        te = fe + trading_window
        print(f'Window {wid:02d}  Form:{dates[si].date()}-{dates[fe-1].date()}  '
              f'Trade:{dates[fe].date()}-{dates[te-1].date()}')
        fp  = price_pivot.iloc[si:fe]
        tp  = price_pivot.iloc[fe:te]
        sel = select_pairs_for_window(fp, sector_map, top_ns)
        rec = {'window_id':wid,'form_start':dates[si],'form_end':dates[fe-1],
               'trade_start':dates[fe],'trade_end':dates[te-1],'pairs_by_n':sel}
        records.append(rec)
        for n in top_ns:
            pairs = sel.get(n,[])
            col   = f'W{wid:02d}'
            wpnl  = pd.Series(0.0, index=dates)
            if pairs:
                for p in pairs:
                    wpnl = wpnl.add(execute_pair_trades(tp,p,fp)/len(pairs))
            all_pnl[n][col] = wpnl
        si += rolling_window; wid += 1
    print(f'Backtest done: {wid} windows.')
    return all_pnl, records

print('Starting backtest...')
all_pnl, window_records = run_backtesting(price_pivot, sector_map)


Starting backtest...
Window 00  Form:2000-01-03-2000-12-29  Trade:2001-01-02-2001-07-02
Window 01  Form:2000-02-01-2001-01-30  Trade:2001-01-31-2001-07-31
Window 02  Form:2000-03-01-2001-02-28  Trade:2001-03-01-2001-08-28
Window 03  Form:2000-03-29-2001-03-28  Trade:2001-03-29-2001-10-02
Window 04  Form:2000-04-27-2001-04-26  Trade:2001-04-27-2001-10-30
Window 05  Form:2000-05-25-2001-05-24  Trade:2001-05-25-2001-11-28
Window 06  Form:2000-06-23-2001-06-22  Trade:2001-06-25-2001-12-27
Window 07  Form:2000-07-24-2001-07-23  Trade:2001-07-24-2002-01-28
Window 08  Form:2000-08-21-2001-08-20  Trade:2001-08-21-2002-02-26
Window 09  Form:2000-09-19-2001-09-24  Trade:2001-09-25-2002-03-26
Window 10  Form:2000-10-17-2001-10-22  Trade:2001-10-23-2002-04-24
Window 11  Form:2000-11-14-2001-11-19  Trade:2001-11-20-2002-05-22
Window 12  Form:2000-12-13-2001-12-18  Trade:2001-12-19-2002-06-20
Window 13  Form:2001-01-12-2002-01-17  Trade:2002-01-18-2002-07-19
Window 14  Form:2001-02-12-2002-02-15  Tr

### 階段四：績效評估與對比分析

| 指標 | 公式 |
|------|------|
| **CAGR** | $(1+r_{total})^{252/T}-1$ |
| **Sharpe** | $\\mu_{ann}/\\sigma_{ann}$ |
| **MDD** | $\\max(1-NAV_t/peak_t)$ |
| **Sortino** | 使用下行標準差 |


In [8]:
def compute_metrics(ret, label=''):
    """Annualised performance metrics from daily return series."""
    r = ret.dropna().replace([np.inf,-np.inf], np.nan).dropna()
    T = len(r)
    if T == 0: return {}
    cum    = (1+r).cumprod()
    tot    = cum.iloc[-1]-1
    cagr   = (1+tot)**(252/T)-1
    vol    = r.std()*np.sqrt(252)
    sharpe = (r.mean()*252)/vol if vol>0 else np.nan
    mdd    = ((cum-cum.cummax())/cum.cummax()).min()
    dn     = r[r<0].std()*np.sqrt(252)
    sortino= (r.mean()*252)/dn if dn>0 else np.nan
    return {'策略':label,'CAGR(%)':round(cagr*100,2),
            '年化波動(%)':round(vol*100,2),
            'Sharpe':round(sharpe,3) if not np.isnan(sharpe) else np.nan,
            'Sortino':round(sortino,3) if not np.isnan(sortino) else np.nan,
            '最大回撤(%)':round(mdd*100,2),
            '勝率(%)':round((r>0).mean()*100,2),
            '總報酬(%)':round(tot*100,2)}

strat_ret = {}
for n in TOP_N_PAIRS:
    df = all_pnl[n]
    if df.empty: continue
    combined = df.sum(axis=1)
    aw = (df!=0).any(axis=0).sum()
    if aw>0: combined /= aw
    strat_ret[f'Top-{n}'] = combined
    
# strat_ret['S&P500(EW)'] = price_pivot.pct_change().mean(axis=1)
# [優化 3] 修正大盤基準的計算異常
pct_change = price_pivot.pct_change()
# 將無限大替換為 NaN
pct_change = pct_change.replace([np.inf, -np.inf], np.nan)
# 排除單日暴漲超過 100% 的極端錯誤數據 (通常是除權息未還原導致)
pct_change = pct_change.where(pct_change < 1.0, np.nan) 

strat_ret['S&P500(EW)'] = pct_change.mean(axis=1)

metrics_df = pd.DataFrame(
    [m for lbl,r in strat_ret.items() for m in [compute_metrics(r,lbl)] if m]
).set_index('策略')
display(metrics_df.style.background_gradient(cmap='RdYlGn',axis=0))


,CAGR(%),年化波動(%),Sharpe,Sortino,最大回撤(%),勝率(%),總報酬(%)
策略,,,,,,,
Top-5,-0.590000,0.110000,-5.451000,-6.888000,-14.340000,27.680000,-14.330000
Top-20,-0.520000,0.090000,-5.724000,-7.454000,-12.740000,25.570000,-12.720000
S&P500(EW),11.900000,20.340000,0.655000,0.841000,-53.150000,53.920000,1748.000000


### 累積 NAV 曲線與回撤圖


In [9]:
COLORS = px.colors.qualitative.Plotly
fig_p = make_subplots(rows=2,cols=1,shared_xaxes=True,
    row_heights=[0.65,0.35],
    subplot_titles=['累積淨値（NAV）曲線','歷史回撤（Drawdown%）'],
    vertical_spacing=0.08)
for i,(lbl,ret) in enumerate(strat_ret.items()):
    r=ret.dropna(); nav=(1+r).cumprod()
    dd=((nav-nav.cummax())/nav.cummax())*100
    c=COLORS[i%len(COLORS)]
    fig_p.add_trace(go.Scatter(x=nav.index,y=nav.values,name=lbl,
        line=dict(color=c,width=2),
        hovertemplate=f'%{{x|%Y-%m-%d}}<br>NAV:%{{y:.4f}}<extra>{lbl}</extra>'),row=1,col=1)
    fig_p.add_trace(go.Scatter(x=dd.index,y=dd.values,name=lbl,showlegend=False,
        fill='tozeroy',line=dict(color=c,width=1),
        hovertemplate=f'%{{x|%Y-%m-%d}}<br>DD:%{{y:.2f}}%<extra>{lbl}</extra>'),row=2,col=1)
fig_p.update_layout(title='配對交易策略績效對比',height=650,template='plotly_dark',
    hovermode='x unified',legend=dict(orientation='h',y=1.02,x=1,xanchor='right'))
fig_p.show()


### Top-5 vs Top-20 統計表（仿 Gatev 2006）


In [10]:
rows_d=[]
for rec in window_records:
    wid=rec['window_id']; col=f'W{wid:02d}'
    row={'視窗':wid,'形成期起':rec['form_start'].date(),'形成期迦':rec['form_end'].date(),
         '交易期起':rec['trade_start'].date(),'交易期迦':rec['trade_end'].date()}
    for n in TOP_N_PAIRS:
        df=all_pnl.get(n,pd.DataFrame())
        if col in df.columns:
            row[f'Top-{n}報酬(%)'] = round(df[col].sum()*100,4)
            row[f'Top-{n}配對數'] = len(rec['pairs_by_n'].get(n,[]))
    row['Top-5配對'] = ', '.join(f"{p['stock_a']}/{p['stock_b']}" for p in rec['pairs_by_n'].get(5,[])[:5])
    rows_d.append(row)
cmp_df = pd.DataFrame([compute_metrics(strat_ret.get(f'Top-{n}',pd.Series()),f'Top-{n}') for n in TOP_N_PAIRS]).set_index('策略')
print('\n=== Top-N Portfolio Comparison ===')
display(cmp_df)
display(pd.DataFrame(rows_d))




=== Top-N Portfolio Comparison ===


,CAGR(%),年化波動(%),Sharpe,Sortino,最大回撤(%),勝率(%),總報酬(%)
策略,,,,,,,
Top-5,-0.5900,0.1100,-5.4510,-6.8880,-14.3400,27.6800,-14.3300
Top-20,-0.5200,0.0900,-5.7240,-7.4540,-12.7400,25.5700,-12.7200


,視窗,形成期起,形成期迦,交易期起,交易期迦,Top-5報酬(%),Top-5配對數,Top-20報酬(%),Top-20配對數,Top-5配對
0,0,2000-01-03,2000-12-29,2001-01-02,2001-07-02,-17.6962,5,-7.4552,20,"BRK-B/WFC, AXP/SPGI, AFL/ALL, BRO/PNC, AIG/BK"
1,1,2000-02-01,2001-01-30,2001-01-31,2001-07-31,19.1104,5,9.6505,20,"AIG/BK, BRK-B/WFC, BRK-B/FITB, BRK-B/MCO, ACGL..."
2,2,2000-03-01,2001-02-28,2001-03-01,2001-08-28,-13.2229,5,-9.7545,20,"BRK-B/WFC, BRK-B/TFC, MCO/WFC, AIG/BK, MRSH/NTRS"
3,3,2000-03-29,2001-03-28,2001-03-29,2001-10-02,-9.6180,5,-6.7465,20,"MCO/WFC, BRK-B/WFC, MRSH/NTRS, FITB/KEY, AIG/NTRS"
4,4,2000-04-27,2001-04-26,2001-04-27,2001-10-30,-7.5063,5,0.7983,20,"BRK-B/WFC, AIG/NTRS, MRSH/NTRS, FISV/SPGI, C/GS"
...,...,...,...,...,...,...,...,...,...,...
304,304,2024-03-05,2025-03-06,2025-03-07,2025-09-05,-15.7443,5,-14.4099,20,"CME/MA, ICE/MRSH, IVZ/MET, KEY/USB, IVZ/USB"
305,305,2024-04-03,2025-04-03,2025-04-04,2025-10-03,-15.9516,5,-20.4056,20,"KEY/RF, CBOE/HIG, PNC/RF, KEY/USB, ALL/TRV"
306,306,2024-05-01,2025-05-02,2025-05-05,2025-10-31,-3.2724,5,-7.9396,20,"KEY/RF, KEY/USB, KEY/PNC, CB/CBOE, CBOE/HIG"
307,307,2024-05-30,2025-06-02,2025-06-03,2025-12-01,1.7576,5,1.9548,20,"KEY/PNC, KEY/RF, ALL/TRV, KEY/USB, GS/NTRS"


## 5.未來工作

### 特徵工程與動態分群：
這個階段的目標是解決傳統靜態配對容易遭遇的「結構漂移」問題，建立具備抗震能力的動態配對池。
●擴充進階特徵：在現有的對數報酬率、波動率（如 ATR）與 VIX 總體經濟數據之上，加入能衡量均值回歸強度的進階特徵，例如 Hurst 指數、擴單根檢定（ADF Test）統計量與動態半衰期（Half-life） 。

●實施板塊過濾與分群：保留目前實作的「產業板塊過濾（Sector-based filtering）」作為前置處理 。

●接著，導入 Scikit-learn 的 DBSCAN 或 HDBSCAN 進行無監督學習，這類演算法能自動將財報爆雷的「雜訊股票」剔除 。

●動態滾動選股：以「月」為單位執行滾動式動態分群，並從中嚴格挑選出 1 至 3 組具備高度共整合特徵的經典配對，準備餵給強化學習代理人 。

### 強化學習模型建構
此階段將運用深度強化學習（DRL），取代傳統固定的 Z-Score 門檻，實現動態進出場控制。
●定義馬可夫決策過程 (MDP)：狀態空間 (State)：整合 VIX 指數、標準化價差與動態半衰期，賦予模型總體市場風險感知能力 。

●動作空間 (Action)：設定為離散空間（做多價差、放空價差、平倉/空手）以加速收斂 。

●獎勵函數 (Reward)：設計扣除交易成本的風險感知獎勵（如夏普比率），並對巨大未實現損失給予負面懲罰，引導代理人學會停損 。

●模型訓練與局部學習：導入 Stable Baselines3 套件，選擇 DQN 或 PPO 演算法 。

●為避免單一全域模型混淆，實施「局部學習（Local Learning）」架構，為特定特徵群組訓練專屬模型 。

### 系統回測與模型驗證
此階段的重點是防範量化回測最致命的「前視偏誤（Look-ahead Bias）」。
●轉換為事件驅動架構：從目前求快的陣列矩陣運算，全面轉換為客製化的 OpenAI Gym 事件驅動（Event-driven）環境，嚴格模擬真實市場的時間流逝、滑價與手續費摩擦成本 。

●滾動式推進分析：捨棄傳統的 K-Fold 交叉驗證，改採金融學界標準的滾動式推進分析（Walk-Forward Optimization），確保系統在 $t$ 時刻僅使用 $t$ 時刻以前的特徵 。

●消融實驗 (Ablation Study)：量化比較移除 VIX 因子前後的效能差異，以實證特徵設計的學術貢獻 。

### 論文撰寫與系統架構展現
最後階段將聚焦於收斂實證結果，並強化資訊管理領域的學術價值。
●模型可解釋性：加入 SHAP 數值分析，剖析 VIX 或價差等特徵對強化學習代理人決策的實際影響權重，提升論文深度 。

●繪製系統架構：產出涵蓋資料擷取、特徵工程、模型推論與風險控制四大層級的資訊系統軟體架構圖，突顯系統整合上的貢獻 。

●排版：依據元智大學教務處的學位論文格式規範（如 A4 規格、版面邊距、中英文摘要格式）與 APA 參考文獻標準進行最終排版定稿 。
